<a href="https://colab.research.google.com/github/VishnuDwivedi/Immigration-Narrative/blob/main/presidential_debates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
from bs4 import BeautifulSoup
import re
import os
import time

# ── CONFIG ──────────────────────────────────────────────────────────────────
BASE_URL   = "https://www.presidency.ucsb.edu"
INDEX_URL  = "https://www.presidency.ucsb.edu/documents/app-categories/elections-and-transitions/debates"
OUTPUT_DIR = "debates"
YEAR_RANGE = range(2000, 2025)   # 2000 – 2024 inclusive
HEADERS    = {"User-Agent": "Mozilla/5.0"}
SLEEP_SEC  = 1.5                 # polite delay between requests
# ────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ── STEP 1: collect all debate links from the index (handles pagination) ────
def get_all_debate_links():
    """Walk every page of the index and return a list of (title, year, url) tuples."""
    links = []
    page  = 0

    while True:
        url = INDEX_URL if page == 0 else f"{INDEX_URL}?page={page}"
        print(f"  Fetching index page {page}: {url}")
        resp = requests.get(url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(resp.text, "html.parser")

        # Each debate is in a <div class="views-row"> or similar; the link is an <a>
        # inside a field--title or the document title area.
        rows = soup.select("div.views-row")
        if not rows:
            break  # no more pages

        found_any = False
        for row in rows:
            a_tag = row.select_one("span.field-title a") or row.select_one("a")
            if not a_tag:
                continue

            title    = a_tag.get_text(strip=True)
            href     = a_tag.get("href", "")
            full_url = href if href.startswith("http") else BASE_URL + href

            # Extract year from the date field or title
            date_tag = row.select_one("span.date-display-single") or row.select_one(".views-field-created")
            year_str = ""
            if date_tag:
                m = re.search(r"\b(20\d{2}|19\d{2})\b", date_tag.get_text())
                if m:
                    year_str = m.group(1)
            if not year_str:
                m = re.search(r"\b(20\d{2}|19\d{2})\b", title)
                year_str = m.group(1) if m else "unknown"

            if int(year_str) in YEAR_RANGE:
                links.append((title, year_str, full_url))
                found_any = True

        if not found_any:
            break

        page += 1
        time.sleep(SLEEP_SEC)

    return links


# ── STEP 2: scrape a single debate transcript ────────────────────────────────
def scrape_debate(url):
    """Return the plain-text transcript from a single debate page."""
    resp = requests.get(url, headers=HEADERS, timeout=15)
    soup = BeautifulSoup(resp.text, "html.parser")

    # The transcript lives inside <div class="field-docs-content"> on this site
    content_div = (
        soup.select_one("div.field-docs-content")
        or soup.select_one("div.field-items")
        or soup.select_one("div#transcript")
        or soup.select_one("article")
    )

    if not content_div:
        return ""

    # Get all paragraph text
    paragraphs = content_div.find_all("p")
    text_blocks = [p.get_text(" ", strip=True) for p in paragraphs if p.get_text(strip=True)]

    # Fall back to full div text if no <p> tags
    if not text_blocks:
        text_blocks = [content_div.get_text("\n", strip=True)]

    return "\n\n".join(text_blocks)


# ── STEP 3: sanitize filename ─────────────────────────────────────────────────
def safe_filename(title, year):
    clean = re.sub(r'[\\/*?:"<>|]', "", title)   # remove illegal chars
    clean = re.sub(r'\s+', "_", clean.strip())    # spaces → underscores
    clean = clean[:120]                            # cap length
    return f"{year}_{clean}.txt"


# ── MAIN ──────────────────────────────────────────────────────────────────────
def main():
    print("=== Presidential Debate Scraper (2000–2024) ===\n")

    print("Step 1 – Collecting debate links from index...")
    debates = get_all_debate_links()
    print(f"  Found {len(debates)} debates in range.\n")

    if not debates:
        print("No debates found – the site's HTML structure may have changed.")
        print("Try inspecting the page and updating the CSS selectors in get_all_debate_links().")
        return

    print("Step 2 – Scraping and saving transcripts...\n")
    for i, (title, year, url) in enumerate(debates, 1):
        fname = safe_filename(title, year)
        fpath = os.path.join(OUTPUT_DIR, fname)

        if os.path.exists(fpath):
            print(f"  [{i}/{len(debates)}] SKIP (exists): {fname}")
            continue

        print(f"  [{i}/{len(debates)}] Scraping: {title}")
        try:
            text = scrape_debate(url)
            if text:
                with open(fpath, "w", encoding="utf-8") as f:
                    f.write(f"TITLE : {title}\n")
                    f.write(f"YEAR  : {year}\n")
                    f.write(f"URL   : {url}\n")
                    f.write("=" * 70 + "\n\n")
                    f.write(text)
                print(f"    ✓ Saved: {fname} ({len(text):,} chars)")
            else:
                print(f"    ✗ No content found at {url}")
        except Exception as e:
            print(f"    ✗ Error: {e}")

        time.sleep(SLEEP_SEC)

    print(f"\nDone! Transcripts saved in ./{OUTPUT_DIR}/")


if __name__ == "__main__":
    main()

=== Presidential Debate Scraper (2000–2024) ===

Step 1 – Collecting debate links from index...
  Fetching index page 0: https://www.presidency.ucsb.edu/documents/app-categories/elections-and-transitions/debates
  Fetching index page 1: https://www.presidency.ucsb.edu/documents/app-categories/elections-and-transitions/debates?page=1
  Fetching index page 2: https://www.presidency.ucsb.edu/documents/app-categories/elections-and-transitions/debates?page=2
  Fetching index page 3: https://www.presidency.ucsb.edu/documents/app-categories/elections-and-transitions/debates?page=3
  Fetching index page 4: https://www.presidency.ucsb.edu/documents/app-categories/elections-and-transitions/debates?page=4
  Fetching index page 5: https://www.presidency.ucsb.edu/documents/app-categories/elections-and-transitions/debates?page=5
  Fetching index page 6: https://www.presidency.ucsb.edu/documents/app-categories/elections-and-transitions/debates?page=6
  Fetching index page 7: https://www.presidency.uc

In [2]:
import os

# List the contents of the 'debates' directory
print(os.listdir("debates"))

# Or, to view the first few lines of a specific file:
# with open("debates/2024_Vice_Presidential_Debate_in_New_York_City.txt", "r", encoding="utf-8") as f:
#     print(f.read(500)) # Print first 500 characters

['2020_Democratic_Candidates_Debate_in_Charleston,_South_Carolina.txt', '2007_Democratic_Presidential_Candidates_Debate_at_South_Carolina_State_University_in_Orangeburg.txt', '2000_Republican_Candidates_Debate_in_Durham,_New_Hampshire.txt', '2016_Republican_Candidates_Debate_in_Houston,_Texas.txt', '2019_Democratic_Candidates_Debate_in_Los_Angeles,_California.txt', '2008_Republican_Presidential_Candidates_Debate_in_Boca_Raton,_Florida.txt', '2007_Republican_Presidential_Candidates_Debate_at_the_University_of_South_Carolina.txt', '2016_Republican_Candidates_Undercard_Debate_in_North_Charleston,_South_Carolina.txt', '2007_Democratic_Presidential_Candidates_Debate_at_Drake_University_in_Des_Moines,_Iowa.txt', '2000_Republican_Presidential_Candidates_Debate_in_Johnston,_Iowa.txt', '2011_Republican_Candidates_Debate_in_Spartanburg,_South_Carolina.txt', '2019_Democratic_Candidates_Debate_in_Houston,_Texas.txt', '2004_Democratic_Presidential_Candidates_Debate_in_Manchester,_New_Hampshire.txt'

In [3]:
import os
import re
from collections import defaultdict

DEBATES_DIR = "/content/debates"

# ── All 145 filenames ────────────────────────────────────────────────────────
all_files = sorted(os.listdir(DEBATES_DIR))

# ── Classification logic ─────────────────────────────────────────────────────
presidential, vice_presidential, republican, democratic = [], [], [], []

for fname in all_files:
    name = fname.replace(".txt", "")

    # VP first (before checking "Presidential" to avoid overlap)
    if re.search(r"Vice.?Presidential", name, re.IGNORECASE):
        vice_presidential.append(fname)

    # General election presidential debates (no party label)
    elif re.match(r"\d{4}_Presidential_Debate", name):
        presidential.append(fname)

    # Republican primaries / candidate debates / forums
    elif re.search(r"Republican", name, re.IGNORECASE):
        republican.append(fname)

    # Democratic primaries / candidate debates / forums
    elif re.search(r"Democratic", name, re.IGNORECASE):
        democratic.append(fname)

    # Edge cases: Palmetto Freedom Forum & Iowa Brown & Black Forum → Republican-aligned
    elif "Palmetto_Freedom_Forum" in name or "Iowa_Brown" in name:
        republican.append(fname)

    else:
        print(f"  ⚠ Unclassified: {fname}")   # safety net


# ── Pretty printer ────────────────────────────────────────────────────────────
def print_category(title, files, emoji):
    print(f"\n{'═'*70}")
    print(f"  {emoji}  {title}  ({len(files)} debates)")
    print(f"{'═'*70}")
    for f in sorted(files):
        year = f[:4]
        label = f[5:].replace(".txt","").replace("_"," ")
        print(f"  {year}  │  {label}")

print_category("PRESIDENTIAL DEBATES (General Election)",  presidential,     "🇺🇸")
print_category("VICE-PRESIDENTIAL DEBATES",                vice_presidential, "🏛️")
print_category("REPUBLICAN CANDIDATES DEBATES & FORUMS",   republican,        "🐘")
print_category("DEMOCRATIC CANDIDATES DEBATES & FORUMS",   democratic,        "🫏")

print(f"\n{'─'*70}")
print(f"  Total classified : {len(presidential)+len(vice_presidential)+len(republican)+len(democratic)}")
print(f"  Presidential     : {len(presidential)}")
print(f"  Vice-Presidential: {len(vice_presidential)}")
print(f"  Republican       : {len(republican)}")
print(f"  Democratic       : {len(democratic)}")
print(f"{'─'*70}")


# ── Build organised dict (useful for topic modelling downstream) ─────────────
debate_lists = {
    "presidential":      sorted(presidential),
    "vice_presidential": sorted(vice_presidential),
    "republican":        sorted(republican),
    "democratic":        sorted(democratic),
}

# ── Save each category list as a .txt index file ─────────────────────────────
os.makedirs("/content/debate_lists", exist_ok=True)
for category, files in debate_lists.items():
    out_path = f"/content/debate_lists/{category}.txt"
    with open(out_path, "w") as f:
        f.write(f"# {category.upper().replace('_',' ')} — {len(files)} debates\n\n")
        for fname in files:
            f.write(fname + "\n")
    print(f"  Saved: {out_path}")

print("\n✓ All 4 category lists saved to /content/debate_lists/")


══════════════════════════════════════════════════════════════════════
  🇺🇸  PRESIDENTIAL DEBATES (General Election)  (19 debates)
══════════════════════════════════════════════════════════════════════
  2000  │  Presidential Debate in Boston
  2000  │  Presidential Debate in St. Louis
  2000  │  Presidential Debate in Winston-Salem, North Carolina
  2004  │  Presidential Debate in Coral Gables, Florida
  2004  │  Presidential Debate in St. Louis, Missouri
  2004  │  Presidential Debate in Tempe, Arizona
  2008  │  Presidential Debate at Belmont University in Nashville, Tennessee
  2008  │  Presidential Debate at Hofstra University in Hempstead, New York
  2008  │  Presidential Debate at the University of Mississippi in Oxford
  2012  │  Presidential Debate in Boca Raton, Florida
  2012  │  Presidential Debate in Denver, Colorado
  2012  │  Presidential Debate in Hempstead, New York
  2016  │  Presidential Debate at Hofstra University in Hempstead, New York
  2016  │  Presidential Deb

### Download All Project Files

First, you need to download all the files generated during this session, including the raw debate transcripts, the categorized debate lists, and the topic modeling results. The topic modeling results were already downloaded as a `.zip` file in the last step. Below are steps to download the rest.

In [20]:
import shutil
from google.colab import files
import os

# Zip the raw debate transcripts
shutil.make_archive("/content/all_debates_transcripts", "zip", "/content/debates")
files.download("/content/all_debates_transcripts.zip")

# Zip the debate category lists
shutil.make_archive("/content/debate_category_lists", "zip", "/content/debate_lists")
files.download("/content/debate_category_lists.zip")

print("\n✓ All debate transcripts and category lists zipped and downloaded.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ All debate transcripts and category lists zipped and downloaded.


### Download the Google Colab Notebook

To download this notebook itself, go to `File > Download > Download .ipynb`.

### Transferring to VS Code / GitHub

Once you have downloaded all the `.zip` files and the `.ipynb` notebook file, follow these steps:

1.  **Create a New Folder Locally**: Create a new folder on your computer for this project (e.g., `presidential_debates_analysis`).
2.  **Extract Downloaded Files**: Unzip the downloaded archives (`presidential_topic_results.zip`, `all_debates_transcripts.zip`, `debate_category_lists.zip`) into this new folder. You'll likely want to organize them into subfolders like `debates`, `debate_lists`, and `topic_results`.
3.  **Place the Notebook**: Move the downloaded `.ipynb` notebook file into the project folder.
4.  **Open in VS Code**: Open VS Code and then open this new project folder.
5.  **Install Jupyter Extension**: If you don't have it, install the 'Jupyter' extension in VS Code to run `.ipynb` notebooks.
6.  **Install Dependencies**: Open a terminal in VS Code (Ctrl+` or View > Terminal) and install the Python libraries used in the notebook:
    ```bash
    pip install requests beautifulsoup4 pandas numpy plotly nltk bertopic[visualization] sentence-transformers umap-learn hdbscan
    ```
7.  **Run the Notebook**: You should now be able to open and run the `.ipynb` notebook directly in VS Code.

### For GitHub:

If you want to push this project to GitHub (after setting it up locally in VS Code):

1.  **Initialize Git**: In the VS Code terminal, navigate to your project folder and initialize a Git repository:
    ```bash
    git init
    ```
2.  **Add Files**: Add all your project files to the Git repository:
    ```bash
    git add .
    ```
3.  **Commit Changes**: Commit your changes:
    ```bash
    git commit -m "Initial commit of presidential debate analysis project"
    ```
4.  **Create GitHub Repository**: Go to GitHub and create a new empty repository (do NOT initialize it with a README, license, or .gitignore).
5.  **Link and Push**: Copy the commands GitHub provides to link your local repository to the remote one and push your changes. It will look something like this:
    ```bash
    git remote add origin https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git
    git branch -M main
    git push -u origin main
    ```
    (Replace `YOUR_USERNAME` and `YOUR_REPO_NAME` with your actual GitHub details.)

This will upload your entire project, including the notebook and all data, to your GitHub repository.

In [4]:
!pip install bertopic[visualization] sentence-transformers umap-learn hdbscan plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.6 MB/s eta 0:00:00


Next steps are loading the packages

In [5]:
import os, re, nltk
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

nltk.download("stopwords", quiet=True)
nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)

DEBATES_DIR  = "/content/debates"
LIST_FILE    = "/content/debate_lists/presidential.txt"

# Stop words: generic English + debate noise
STOP_WORDS = set(stopwords.words("english")) | {
    "applause", "laughter", "audience", "moderator", "question", "answer",
    "mr", "ms", "senator", "governor", "secretary", "vice", "president",
    "um", "uh", "yeah", "okay", "well", "going", "think", "know", "said",
    "say", "people", "time", "want", "need", "make", "look", "come", "got",
    "get", "let", "way", "thing", "right", "good", "great", "would",
    "could", "should", "also", "like", "just", "really", "actually",
    "that", "this", "there", "their", "they", "what", "when", "where",
    "lehrer", "schieffer", "ifill", "raddatz", "holt", "wallace",  # moderator names
}


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [6]:
# ── CELL 3: Load only presidential debate files ──────────────────
def load_presidential_debates(list_file, debates_dir):
    # Read the index file
    with open(list_file) as f:
        filenames = [
            line.strip() for line in f
            if line.strip() and not line.startswith("#")
        ]

    records = []
    for fname in filenames:
        fpath = os.path.join(debates_dir, fname)
        if not os.path.exists(fpath):
            print(f"  ⚠ Missing: {fname}")
            continue

        with open(fpath, "r", encoding="utf-8") as f:
            raw = f.read()

        # Strip scraper header
        body = re.sub(r"^(TITLE|YEAR|URL)\s*:.*\n", "", raw, flags=re.MULTILINE)
        body = re.sub(r"^={10,}\n", "",               body, flags=re.MULTILINE).strip()

        year_m  = re.search(r"YEAR\s*:\s*(\d{4})", raw)
        title_m = re.search(r"TITLE\s*:\s*(.+)",   raw)
        year    = int(year_m.group(1))  if year_m  else int(fname[:4])
        title   = title_m.group(1).strip() if title_m else fname

        records.append({
            "year":  year,
            "title": title,
            "text":  body,
            "file":  fname
        })

    df = pd.DataFrame(records).sort_values("year").reset_index(drop=True)
    print(f"✓ Loaded {len(df)} presidential debates  |  {df['year'].min()}–{df['year'].max()}")
    print(df[["year", "title"]].to_string(index=False))
    return df

debates_df = load_presidential_debates(LIST_FILE, DEBATES_DIR)


✓ Loaded 19 presidential debates  |  2000–2024
 year                                                                     title
 2000                                             Presidential Debate in Boston
 2000                                          Presidential Debate in St. Louis
 2000                      Presidential Debate in Winston-Salem, North Carolina
 2004                              Presidential Debate in Coral Gables, Florida
 2004                                Presidential Debate in St. Louis, Missouri
 2004                                     Presidential Debate in Tempe, Arizona
 2008         Presidential Debate at Belmont University in Nashville, Tennessee
 2008          Presidential Debate at Hofstra University in Hempstead, New York
 2008            Presidential Debate at the University of Mississippi in Oxford
 2012                                Presidential Debate in Boca Raton, Florida
 2012                                   Presidential Debate in Denver, Co

In [7]:
# ── CELL 4: Sentence-level chunking ─────────────────────────────
# BERTopic needs SHORT documents — sentences work best
def clean(text):
    text = re.sub(r"\[.*?\]|\(.*?\)", " ", text)  # stage directions
    text = re.sub(r"[^a-zA-Z\s']",    " ", text)  # keep letters only
    return re.sub(r"\s+", " ", text).strip().lower()

def to_sentences(row, min_meaningful_words=8):
    out = []
    for s in sent_tokenize(row["text"]):
        c     = clean(s)
        words = [w for w in c.split() if w not in STOP_WORDS and len(w) > 2]
        if len(words) >= min_meaningful_words:
            out.append({"year": row["year"], "title": row["title"], "sentence": c})
    return out

rows = []
for _, r in debates_df.iterrows():
    rows.extend(to_sentences(r))

sdf = pd.DataFrame(rows)
print(f"\n✓ {len(sdf):,} sentences across {sdf['year'].nunique()} debate years")
print(sdf.groupby("year").size().rename("sentences").to_string())




✓ 5,383 sentences across 7 debate years
year
2000    859
2004    825
2008    909
2012    961
2016    800
2020    504
2024    525


In [8]:
# ── CELL 5: BERTopic model ───────────────────────────────────────
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# With only 19 debates, use smaller cluster sizes so we don't lose topics
umap_model = UMAP(
    n_neighbors=10,    # smaller = more local structure captured
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=15,   # lower than default — fewer docs per year
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

vectorizer = CountVectorizer(
    stop_words=list(STOP_WORDS),
    ngram_range=(1, 2),    # captures "climate change", "tax cuts", "foreign policy"
    min_df=3
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
    nr_topics="auto",
    top_n_words=10,
    verbose=True
)

docs   = sdf["sentence"].tolist()
years  = sdf["year"].tolist()

topics, probs = topic_model.fit_transform(docs)
sdf["topic"] = topics

n_topics = len(set(topics)) - 1   # -1 is noise bucket
print(f"\n✓ Discovered {n_topics} topics  (topic -1 = noise/outliers)")



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-03-13 03:59:18,670 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/169 [00:00<?, ?it/s]

2026-03-13 04:01:06,708 - BERTopic - Embedding - Completed ✓
2026-03-13 04:01:06,709 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-13 04:01:43,410 - BERTopic - Dimensionality - Completed ✓
2026-03-13 04:01:43,413 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-13 04:01:43,806 - BERTopic - Cluster - Completed ✓
2026-03-13 04:01:43,809 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-03-13 04:01:44,239 - BERTopic - Representation - Completed ✓
2026-03-13 04:01:44,241 - BERTopic - Topic reduction - Reducing number of topics
2026-03-13 04:01:44,279 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-13 04:01:44,686 - BERTopic - Representation - Completed ✓
2026-03-13 04:01:44,691 - BERTopic - Topic reduction - Reduced number of topics from 86 to 40



✓ Discovered 39 topics  (topic -1 = noise/outliers)


In [9]:
# ── CELL 6: Topic labels ────────────────────────────────────────
topic_info = topic_model.get_topic_info()

# Clean label: top 4 keywords joined
def make_label(topic_id):
    words = topic_model.get_topic(topic_id)
    if not words:
        return "noise"
    return " | ".join([w for w, _ in words[:4]])

topic_label_map = {
    row["Topic"]: make_label(row["Topic"])
    for _, row in topic_info.iterrows()
    if row["Topic"] != -1
}

sdf["topic_label"] = sdf["topic"].map(topic_label_map).fillna("noise")

print("\nTop Topics Discovered:")
for tid, lbl in sorted(topic_label_map.items()):
    count = (sdf["topic"] == tid).sum()
    print(f"  Topic {tid:>3}: {lbl:<50}  ({count} sentences)")




Top Topics Discovered:
  Topic   0: jobs | tax | mccain | oil                           (1304 sentences)
  Topic   1: troops | israel | afghanistan | bin laden           (644 sentences)
  Topic   2: insurance | medicare | seniors | obamacare          (233 sentences)
  Topic   3: court | gun | supreme court | supreme               (215 sentences)
  Topic   4: biden | welker | street | wall street               (118 sentences)
  Topic   5: clinton | nafta | husband | hillary                 (118 sentences)
  Topic   6: social security | social | security | trust         (93 sentences)
  Topic   7: iran | nuclear | sanctions | nuclear weapons        (87 sentences)
  Topic   8: abortion | birth | baby | month                     (80 sentences)
  Topic   9: ronald reagan | ronald | reagan | served            (72 sentences)
  Topic  10: immigration | border | illegally | across border    (63 sentences)
  Topic  11: china | steel | goods | day one                     (61 sentences)
  Topic  

In [10]:
# Look at the top 12 most frequent topics
topic_info = topic_model.get_topic_info()
display(topic_info.head(12))

# Let's visualize the top 12 topics to easily read their keywords
chart = topic_model.visualize_barchart(top_n_topics=12, n_words=8)
chart.show()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1605,-1_health_health care_care_year,"[health, health care, care, year, trump, war, ...",[but we're going to have to invest in the amer...
1,0,1304,0_jobs_tax_mccain_oil,"[jobs, tax, mccain, oil, energy, debt, romney,...",[we have to have energy independence so i've p...
2,1,644,1_troops_israel_afghanistan_bin laden,"[troops, israel, afghanistan, bin laden, laden...",[i was involved in a number of efforts to take...
3,2,233,2_insurance_medicare_seniors_obamacare,"[insurance, medicare, seniors, obamacare, insu...",[million people have lost their private insura...
4,3,215,3_court_gun_supreme court_supreme,"[court, gun, supreme court, supreme, roe, amen...",[and what i did is i put three great supreme c...
5,4,118,4_biden_welker_street_wall street,"[biden, welker, street, wall street, wall, pen...",[welker ok very quickly and then we're gonna h...
6,5,118,5_clinton_nafta_husband_hillary,"[clinton, nafta, husband, hillary, signed, yor...",[the nafta deal signed by her husband is one o...
7,6,93,6_social security_social_security_trust,"[social security, social, security, trust, you...",[the governor wants to divert out of every off...
8,7,87,7_iran_nuclear_sanctions_nuclear weapons,"[iran, nuclear, sanctions, nuclear weapons, th...",[mccain my reading of the threat from iran is ...
9,8,80,8_abortion_birth_baby_month,"[abortion, birth, baby, month, women, ban, wom...",[first of all on the issue of partial birth or...


In [11]:
# ── CELL 8: Per-election-year heatmap ───────────────────────────
pivot = (
    sdf[sdf["topic"] != -1]
    .groupby(["year", "topic_label"])
    .size()
    .reset_index(name="count")
)
pivot["pct"] = pivot.groupby("year")["count"].transform(
    lambda x: x / x.sum() * 100
)

heat = pivot.pivot(index="year", columns="topic_label", values="pct").fillna(0)

fig2 = px.imshow(
    heat,
    aspect="auto",
    color_continuous_scale="Blues",
    title="Topic Share Per Election Year — Presidential Debates (%)",
    labels={"x": "Topic", "y": "Year", "color": "% sentences"},
    height=500
)
fig2.update_xaxes(tickangle=40, tickfont_size=10)
fig2.show()



In [12]:
top6 = (
    pivot.sort_values(["year", "pct"], ascending=[True, False])
    .groupby("year").head(6)
)

fig3 = px.bar(
    top6,
    x="pct", y="topic_label",
    color="topic_label",
    facet_col="year", facet_col_wrap=4,
    orientation="h",
    title="Top 6 Topics Each Presidential Debate Year",
    labels={"pct": "% of sentences", "topic_label": "Topic"},
    height=800
)
fig3.update_layout(showlegend=False)
fig3.update_yaxes(matches=None)   # each facet has its own y-axis
fig3.show()

In [13]:
# ── CELL 10: Topic rise & fall tracker ──────────────────────────
THRESHOLD = 2.0   # topic must hit 2% in a year to count as "active"

timeline = []
for lbl in heat.columns:
    active = heat.index[heat[lbl] >= THRESHOLD].tolist()
    if not active:
        continue
    timeline.append({
        "topic":      lbl,
        "first_year": min(active),
        "last_year":  max(active),
        "peak_year":  int(heat[lbl].idxmax()),
        "peak_pct":   round(float(heat[lbl].max()), 2),
    })

timeline_df = pd.DataFrame(timeline).sort_values("peak_pct", ascending=False)

print("\nTopic Rise & Fall Summary:")
print(timeline_df.to_string(index=False))

fig4 = px.timeline(
    timeline_df.assign(
        start=lambda d: pd.to_datetime(d["first_year"].astype(str) + "-01-01"),
        end  =lambda d: pd.to_datetime(d["last_year"].astype(str)  + "-12-31"),
    ),
    x_start="start", x_end="end",
    y="topic",
    color="peak_pct",
    color_continuous_scale="Viridis",
    title="When Each Topic Was Active — Presidential Debates",
    labels={"peak_pct": "Peak %"},
    height=600
)
fig4.show()


Topic Rise & Fall Summary:
                                             topic  first_year  last_year  peak_year  peak_pct
                         jobs | tax | mccain | oil        2000       2024       2012     49.73
         troops | israel | afghanistan | bin laden        2000       2024       2004     25.43
             biden | welker | street | wall street        2020       2024       2020     20.55
               clinton | nafta | husband | hillary        2016       2024       2016     14.66
             court | gun | supreme court | supreme        2000       2024       2016     11.17
        insurance | medicare | seniors | obamacare        2000       2024       2000      8.88
                   abortion | birth | baby | month        2008       2024       2024      8.76
       social security | social | security | trust        2000       2004       2000      7.01
                      gore | hate | crimes | james        2000       2000       2000      6.07
             sir | wai

In [14]:
# ── CELL 11: Semantic topic map ──────────────────────────────────
fig5 = topic_model.visualize_topics(
    title="Topic Similarity Map — Presidential Debates"
)
fig5.show()

In [15]:





# ── CELL 12: Election-cycle comparison ──────────────────────────
# Group years into election cycles for a cleaner comparison
cycle_map = {
    2000: "2000 (Bush vs Gore)",
    2004: "2004 (Bush vs Kerry)",
    2008: "2008 (Obama vs McCain)",
    2012: "2012 (Obama vs Romney)",
    2016: "2016 (Trump vs Clinton)",
    2020: "2020 (Biden vs Trump)",
    2024: "2024 (Harris vs Trump)",
}
sdf["cycle"] = sdf["year"].map(cycle_map).fillna(sdf["year"].astype(str))

cycle_pivot = (
    sdf[sdf["topic"] != -1]
    .groupby(["cycle", "topic_label"])
    .size()
    .reset_index(name="count")
)
cycle_pivot["pct"] = cycle_pivot.groupby("cycle")["count"].transform(
    lambda x: x / x.sum() * 100
)

# Top 8 topics per cycle
top8_cycle = (
    cycle_pivot.sort_values(["cycle", "pct"], ascending=[True, False])
    .groupby("cycle").head(8)
)

fig6 = px.bar(
    top8_cycle,
    x="topic_label", y="pct",
    color="topic_label",
    facet_row="cycle",
    title="Top 8 Topics Per Election Cycle — Presidential Debates",
    labels={"pct": "%", "topic_label": "Topic"},
    height=1400
)
fig6.update_layout(showlegend=False)
fig6.update_xaxes(tickangle=40, tickfont_size=9)
fig6.update_yaxes(matches=None)
fig6.show()




In [19]:
# ── CELL 13: Save & download results ────────────────────────────
import shutil
from google.colab import files
topics_over_time = topic_model.topics_over_time(docs, years)
print("\n✓ Calculated topics over time.")

os.makedirs("/content/presidential_topic_results", exist_ok=True)

sdf.to_csv("/content/presidential_topic_results/sentences_with_topics.csv",   index=False)
topic_info.to_csv("/content/presidential_topic_results/topic_info.csv",       index=False)
topics_over_time.to_csv("/content/presidential_topic_results/topics_over_time.csv", index=False)
timeline_df.to_csv("/content/presidential_topic_results/topic_timeline.csv",  index=False)

shutil.make_archive("/content/presidential_topic_results", "zip",
                    "/content/presidential_topic_results")
files.download("/content/presidential_topic_results.zip")

print("✓ All results saved and downloaded!")


7it [00:00, 15.08it/s]



✓ Calculated topics over time.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ All results saved and downloaded!


In [ ]:
topics_over_time = topic_model.topics_over_time(docs, years)
print("\n✓ Calculated topics over time.")